In [ ]:
from pathlib import Path
import shutil

import polars as pl
import yaml

from build_commuting_aware_visitation_network import (
    determine_visitation_network,
    aggregate_by_device,
    aggregate_by_trip,
    filter_connections,
)


In [2]:
with open("../config.budapest.yaml", "r") as fp:
    config = yaml.safe_load(fp)

with open("09_visitation_network_config.yaml", "r") as fp:
    vn_config = yaml.safe_load(fp)


path = f"{config['dirs']['output']}/{config['city']}"

In [3]:
for res in range(7, 11):
    for by in ["device", "trip"]:
        hw = pl.read_parquet(
            f"{path}/homes/{vn_config['stop_detection_variant']}/home_work_h3_{res}.parquet"
        )
        hw = hw.rename({f"h3_{res}_home": "home", f"h3_{res}_work": "work"})
        hw = hw.drop_nulls()

        hw = hw.filter(
            (pl.col("count_home") > vn_config["filtering"]["minimum_stops_for_home"])
            & (pl.col("count_work") > vn_config["filtering"]["minimum_stops_for_work"])
        )

        connections = pl.read_parquet(
            f"{path}/hexagon_network/{vn_config['stop_detection_variant']}/{res}/connections.parquet"
        )
        connections = filter_connections(connections, vn_config["filtering"])
        visitations = determine_visitation_network(connections, hw, res=res)
        aggregated = aggregate_by_trip(visitations, hw)
        if by == "device":
            aggregated = aggregate_by_device(visitations, hw)

        aggregated = aggregated.sort(["home", "work", "source", "target"])

        out_path = f"{path}/visitation_network/{vn_config['home_detection']}/{vn_config['stop_detection_variant']}/{by}/{vn_config['variant']}"
        Path(out_path).mkdir(exist_ok=True, parents=True)
        shutil.copy(
            "09_visitation_network_config.yaml",
            f"{out_path}/visitation_network_config.yaml",
        )
        aggregated.write_csv(f"{out_path}/{res}.csv")
        aggregated.write_parquet(f"{out_path}/{res}.parquet")

# runs 1m 45s